# Medical Appointments Dataset: Cleaning Notebook

This notebook cleans `dirty_dataset.csv` (medical appointment no-show data) and writes:

- `cleaned_dataset.csv`: the cleaned data
- `removed_rows.csv`: every row removed, with the reason

**Steps:** load → profile → rename columns → fix patient IDs → fix types → fix text → remove invalid rows → validate → save

In [ ]:
import pandas as pd
from decimal import Decimal

INPUT_FILE = "dirty_dataset.csv"
MAX_PLAUSIBLE_AGE = 110

## 1. Load
Everything is read as text first so nothing (like long patient IDs) gets silently altered.

In [ ]:
raw = pd.read_csv(INPUT_FILE, dtype=str, keep_default_na=False)
df = raw.copy()
print("Rows:", len(df), "| Columns:", df.shape[1])
df.head()

Rows: 110527 | Columns: 14


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,29872499824296,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,558997776694438,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4262962299951,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,867951213174,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8841186448183,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## 2. Profile the problems

In [ ]:
print("Blank cells per column:")
print((df.apply(lambda s: s.str.strip() == "")).sum().to_string())
print("\nExact duplicate rows:", df.duplicated().sum())

age = pd.to_numeric(df["Age"], errors="coerce")
print("\nAge < 0:", (age < 0).sum(), "| Age >", MAX_PLAUSIBLE_AGE, ":", (age > MAX_PLAUSIBLE_AGE).sum())

sched = pd.to_datetime(df["ScheduledDay"], utc=True)
appt  = pd.to_datetime(df["AppointmentDay"], utc=True)
print("Appointment date before scheduling date:", (appt.dt.normalize() < sched.dt.normalize()).sum())

print("Patient IDs in scientific notation:", df["PatientId"].str.contains("E", case=False).sum())
df.loc[df["PatientId"].str.contains("E", case=False), ["PatientId", "AppointmentID"]]

Blank cells per column:


PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0

Exact duplicate rows: 0

Age < 0: 1 | Age > 110 : 5


Appointment date before scheduling date: 5
Patient IDs in scientific notation: 5


,PatientId,AppointmentID
3950,9377952927E-5,5712759
73228,53761528476E-5,5637728
73303,14172416655E-5,5637648
100517,3921784439E-5,5751990
105430,4374175652E-5,5760144


## 3. Rename columns
Fix the misspellings (`Hipertension`, `Handcap`) and use consistent snake_case.

In [ ]:
df = df.rename(columns={
    "PatientId": "patient_id",
    "AppointmentID": "appointment_id",
    "Gender": "gender",
    "ScheduledDay": "scheduled_at",
    "AppointmentDay": "appointment_date",
    "Age": "age",
    "Neighbourhood": "neighbourhood",
    "Scholarship": "scholarship",
    "Hipertension": "hypertension",
    "Diabetes": "diabetes",
    "Alcoholism": "alcoholism",
    "Handcap": "handicap_level",
    "SMS_received": "sms_received",
    "No-show": "no_show",
})
df.columns.tolist()

['patient_id',
 'appointment_id',
 'gender',
 'scheduled_at',
 'appointment_date',
 'age',
 'neighbourhood',
 'scholarship',
 'hypertension',
 'diabetes',
 'alcoholism',
 'handicap_level',
 'sms_received',
 'no_show']

## 4. Fix patient IDs stored in scientific notation
Values like `9377952927E-5` are converted to plain decimal text (`93779.52927`). `Decimal` is used so no float rounding occurs. IDs stay as text because they are identifiers, not numbers.

In [ ]:
sci = df["patient_id"].str.contains("E", case=False)
df.loc[sci, "patient_id"] = df.loc[sci, "patient_id"].map(lambda v: format(Decimal(v), "f"))
print("Fixed", sci.sum(), "IDs:", df.loc[sci, "patient_id"].tolist())

Fixed 5 IDs: ['93779.52927', '537615.28476', '141724.16655', '39217.84439', '43741.75652']


## 5. Fix data types
- `scheduled_at` → timestamp (drops the stray `T`/`Z` formatting)
- `appointment_date` → date only (the time was always midnight)
- numeric and flag columns → integers

In [ ]:
df["scheduled_at"] = pd.to_datetime(df["scheduled_at"], utc=True).dt.tz_localize(None)
df["appointment_date"] = pd.to_datetime(df["appointment_date"], utc=True).dt.tz_localize(None).dt.normalize()

int_cols = ["appointment_id", "age", "scholarship", "hypertension",
            "diabetes", "alcoholism", "handicap_level", "sms_received"]
df[int_cols] = df[int_cols].astype(int)
df.dtypes

patient_id                     str
appointment_id               int64
gender                         str
scheduled_at        datetime64[us]
appointment_date    datetime64[us]
age                          int64
neighbourhood                  str
scholarship                  int64
hypertension                 int64
diabetes                     int64
alcoholism                   int64
handicap_level               int64
sms_received                 int64
no_show                        str
dtype: object

## 6. Fix text values
`JOANA D´ARC` uses an accent character (´) instead of an apostrophe.

In [ ]:
df["neighbourhood"] = df["neighbourhood"].str.replace("´", "'", regex=False)
df.loc[df["neighbourhood"].str.contains("JOANA"), "neighbourhood"].unique()

<StringArray>
['JOANA D'ARC']
Length: 1, dtype: str

## 7. Remove invalid rows
Removed rows are **logged, not silently dropped**:

| Rule | Why |
|---|---|
| `age < 0` | impossible |
| `age > 110` | implausible (likely a placeholder or entry error) |
| appointment date earlier than scheduling date | impossible |

In [ ]:
reason = pd.Series("", index=df.index)
reason[df["age"] < 0] = "Negative age"
reason[df["age"] > MAX_PLAUSIBLE_AGE] = f"Implausible age (>{MAX_PLAUSIBLE_AGE})"
reason[df["appointment_date"] < df["scheduled_at"].dt.normalize()] = "Appointment date before scheduling date"

bad = reason != ""
removed = df[bad].assign(reason=reason[bad])
clean = df[~bad].sort_values("appointment_id").reset_index(drop=True)

print(f"Original: {len(df)} | Cleaned: {len(clean)} | Removed: {len(removed)}")
print(removed["reason"].value_counts().to_string())
removed

Original: 110527 | Cleaned: 110516 | Removed: 11
reason
Appointment date before scheduling date    5
Implausible age (>110)                     5
Negative age                               1


,patient_id,appointment_id,gender,scheduled_at,appointment_date,age,neighbourhood,scholarship,hypertension,diabetes,alcoholism,handicap_level,sms_received,no_show,reason
27033,7839272661752,5679978,M,2016-05-10 10:51:53,2016-05-09,38,RESISTÊNCIA,0,0,0,0,1,0,Yes,Appointment date before scheduling date
55226,7896293967868,5715660,F,2016-05-18 14:50:41,2016-05-17,19,SANTO ANTÔNIO,0,0,0,0,1,0,Yes,Appointment date before scheduling date
63912,31963211613981,5700278,F,2016-05-16 09:17:44,2016-05-19,115,ANDORINHAS,0,0,0,0,1,0,Yes,Implausible age (>110)
63915,31963211613981,5700279,F,2016-05-16 09:17:44,2016-05-19,115,ANDORINHAS,0,0,0,0,1,0,Yes,Implausible age (>110)
64175,24252258389979,5664962,F,2016-05-05 13:43:58,2016-05-04,22,CONSOLAÇÃO,0,0,0,0,0,0,Yes,Appointment date before scheduling date
68127,31963211613981,5562812,F,2016-04-08 14:29:17,2016-05-16,115,ANDORINHAS,0,0,0,0,1,0,Yes,Implausible age (>110)
71533,998231581612122,5686628,F,2016-05-11 13:49:20,2016-05-05,81,SANTO ANTÔNIO,0,0,0,0,0,0,Yes,Appointment date before scheduling date
72362,3787481966821,5655637,M,2016-05-04 06:50:57,2016-05-03,7,TABUAZEIRO,0,0,0,0,0,0,Yes,Appointment date before scheduling date
76284,31963211613981,5744037,F,2016-05-30 09:44:51,2016-05-30,115,ANDORINHAS,0,0,0,0,1,0,No,Implausible age (>110)
97666,748234579244724,5717451,F,2016-05-19 07:57:56,2016-06-03,115,SÃO JOSÉ,0,1,0,0,0,1,No,Implausible age (>110)


## 8. Validate

In [ ]:
assert clean["appointment_id"].is_unique
assert clean.isna().sum().sum() == 0
assert clean["age"].between(0, MAX_PLAUSIBLE_AGE).all()
assert (clean["appointment_date"] >= clean["scheduled_at"].dt.normalize()).all()
assert clean["gender"].isin(["F", "M"]).all()
assert clean["no_show"].isin(["Yes", "No"]).all()
print("All checks passed")

All checks passed


## 9. Save
Dates are written as `YYYY-MM-DD` (appointment) and `YYYY-MM-DD HH:MM:SS` (scheduled).

In [ ]:
out = clean.copy()
out["scheduled_at"] = out["scheduled_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
out["appointment_date"] = out["appointment_date"].dt.strftime("%Y-%m-%d")
out.to_csv("cleaned_dataset.csv", index=False)

rem = removed.copy()
rem["scheduled_at"] = rem["scheduled_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
rem["appointment_date"] = rem["appointment_date"].dt.strftime("%Y-%m-%d")
rem.to_csv("removed_rows.csv", index=False)

out.head()

,patient_id,appointment_id,gender,scheduled_at,appointment_date,age,neighbourhood,scholarship,hypertension,diabetes,alcoholism,handicap_level,sms_received,no_show
0,832256398961987,5030230,F,2015-11-10 07:13:56,2016-05-04,51,RESISTÊNCIA,0,0,0,0,0,1,No
1,91637474953513,5122866,M,2015-12-03 08:17:28,2016-05-02,34,VILA RUBIM,0,1,0,0,0,1,Yes
2,1216586867796,5134197,F,2015-12-07 10:40:59,2016-06-03,27,SÃO CRISTÓVÃO,1,0,0,0,0,1,Yes
3,31899595421534,5134220,F,2015-12-07 10:42:42,2016-06-03,48,MARUÍPE,0,1,1,0,0,1,No
4,9582232334148,5134223,F,2015-12-07 10:43:01,2016-06-03,80,SÃO CRISTÓVÃO,0,1,1,0,0,1,No
